# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [9]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** **Line chart, indexed to 1970 = 100.** Growth over time is a line-chart question, and indexing each continent to its own 1970 level makes the *relative* growth comparable (raw totals would bury Oceania under Asia). Africa is the only coloured line (deep blue); the rest are light grey, so the eye goes straight to the answer. Names sit at the line ends, so there is no legend to decode.

In [10]:
# Q1
ACCENT, GREY = '#0B5394', '#BDBDBD'

cont = df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
cont['Index'] = cont['Population'] / cont.groupby('Continent')['Population'].transform('first') * 100
last = cont[cont['Year'] == cont['Year'].max()].set_index('Continent')['Index'].sort_values()
fastest = last.index[-1]
cont['Label'] = np.where(cont['Year'] == 2022, cont['Continent'] + ' ' + cont['Index'].round().astype(int).astype(str), '')

fig = px.line(cont, x='Year', y='Index', color='Continent', text='Label', template='plotly_white',
              color_discrete_map={c: ACCENT if c == fastest else GREY for c in last.index},
              title=f"Growth champion: {fastest} is {last[fastest] / 100:.1f}x its 1970 size, while Europe has barely moved ({last['Europe'] / 100:.1f}x)")
fig.update_traces(mode='lines+text', textposition='middle right', line_width=2, cliponaxis=False)
fig.update_traces(line_width=5, textfont=dict(color=ACCENT, size=14), selector=dict(name=fastest))
fig.update_layout(showlegend=False, title_x=0, title_font_size=18, margin=dict(r=140, t=70),
                  xaxis_title=None, yaxis_title='Population, 1970 = 100')
fig.update_xaxes(showgrid=False, range=[1969, 2034], tickvals=[1970, 1980, 1990, 2000, 2010, 2022])
fig.update_yaxes(gridcolor='#EEEEEE', zeroline=False)
fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** **Sorted bar chart (horizontal).** A top-15 ranking is best read as bars sorted by length, and horizontal bars fit the long country names. Instead of colouring by group in a legend, only the African bars are blue and the rest are grey; the subtitle counts them, and values (in millions) are printed on the bars so the x-axis can go.

In [11]:
# Q2
ACCENT, GREY = '#0B5394', '#C8C8C8'

g = df.assign(Gain=(df['2022 Population'] - df['1970 Population']) / 1e6)
top = g.nlargest(15, 'Gain').sort_values('Gain', ascending=False).reset_index(drop=True)
n_af = int((top['Continent'] == 'Africa').sum())

fig = px.bar(top, x='Gain', y='Country', orientation='h', text=top['Gain'].round().astype(int), template='plotly_white',
             title=f"{n_af} of the 15 biggest population gainers since 1970 are in Africa; Asia still dominates the top<br>"
                   f"<sup>People added between 1970 and 2022, millions. African countries in blue.</sup>")
fig.update_traces(marker_color=[ACCENT if c == 'Africa' else GREY for c in top['Continent']],
                  textposition='outside', cliponaxis=False, hovertemplate='%{y}: +%{x:.0f}M<extra></extra>')
fig.update_yaxes(autorange='reversed', title=None)          # biggest on top
fig.update_xaxes(visible=False)
fig.update_layout(title_x=0, title_font_size=18, margin=dict(t=90, r=40), bargap=0.25)
fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** **Scatter plot, one small panel per continent (log-log axes).** Area vs population is two numeric variables, so a scatter plot. Splitting it into continent panels answers "does it differ by continent?" without overplotting 234 dots in one cloud. Each panel shows a faint grey cloud of all countries for reference, the continent's own countries in colour, and a dashed global trend line, so the countries that break the pattern sit visibly far from the line and are labelled.

In [12]:
# Q3
PAL = {'Africa': '#D55E00', 'Asia': '#0072B2', 'Europe': '#CC79A7',
       'North America': '#E69F00', 'South America': '#009E73', 'Oceania': '#555555'}   # Okabe-Ito
PAL['South America'] = '#56B4E9'                                                       # avoid green

d = df.copy()
slope, icpt = np.polyfit(np.log10(d['Area']), np.log10(d['2022 Population']), 1)
d['Resid'] = np.log10(d['2022 Population']) - (icpt + slope * np.log10(d['Area']))
d['Label'] = ''
for _, grp in d.groupby('Continent'):                                   # label the 2 biggest outliers per continent
    d.loc[grp['Resid'].abs().nlargest(2).index, 'Label'] = grp.loc[grp['Resid'].abs().nlargest(2).index, 'Country']

fig = px.scatter(d, x='Area', y='2022 Population', color='Continent', text='Label', hover_name='Country',
                 facet_col='Continent', facet_col_wrap=3, log_x=True, log_y=True, color_discrete_map=PAL,
                 category_orders={'Continent': ['Africa', 'Asia', 'Europe', 'North America', 'South America', 'Oceania']},
                 template='plotly_white', height=650,
                 title="Larger countries have more people everywhere, but Asia sits above the line and Oceania/South America below;<br>"
                       "the labelled countries are the biggest departures from the trend")
fig.update_traces(marker_size=7, textposition='top center', textfont_size=10)

xs = np.array([np.log10(d['Area'].min()), np.log10(d['Area'].max())])
for r in (1, 2):
    for c in (1, 2, 3):
        fig.add_scatter(x=10 ** xs, y=10 ** (icpt + slope * xs), mode='lines', hoverinfo='skip', showlegend=False,
                        line=dict(color='#999999', dash='dash', width=1), row=r, col=c)

fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1], font=dict(size=13)))
fig.update_layout(showlegend=False, title_x=0, title_font_size=17, margin=dict(t=100))
fig.update_xaxes(showgrid=False, title=None)
fig.update_yaxes(showgrid=False, title=None)
fig.add_annotation(text='Land area, km² (log)', xref='paper', yref='paper', x=0.5, y=-0.07, showarrow=False)
fig.add_annotation(text='2022 population (log)', xref='paper', yref='paper', x=-0.05, y=0.5, textangle=-90, showarrow=False)
fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** **Violin plot with an inner box, log axis.** The question asks for the *shape* of the spread, not just summary stats, so violins show where countries cluster (e.g. Africa's and South America's mass at low density, Asia's long tail) while the inner box keeps median and quartiles readable. Density spans 0.03 to 23,000 people/km², so the axis is logarithmic. Continents are sorted by median, one colour each, no legend, and the densest country per continent is labelled.

In [13]:
# Q4
PAL = {'Africa': '#D55E00', 'Asia': '#0072B2', 'Europe': '#CC79A7',
       'North America': '#E69F00', 'South America': '#56B4E9', 'Oceania': '#666666'}

order = df.groupby('Continent')['Density'].median().sort_values().index.tolist()
med = df.groupby('Continent')['Density'].median()
spread = df.groupby('Continent')['Density'].agg(['min', 'max'])

fig = px.violin(df, x='Continent', y='Density', color='Continent', box=True, points=False, log_y=True,
                hover_name='Country', category_orders={'Continent': order}, color_discrete_map=PAL,
                template='plotly_white',
                title=f"The typical continent has about 20 to 220 people per km², but within each one density spans<br>"
                      f"2 to 4 orders of magnitude, from empty plains to city-states")
fig.update_traces(meanline_visible=False, opacity=0.75, width=0.8, box_width=0.12, spanmode='hard')

for cname, idx in df.groupby('Continent')['Density'].idxmax().items():
    fig.add_annotation(x=cname, y=np.log10(df.loc[idx, 'Density']), text=f"{df.loc[idx, 'Country']}<br>{df.loc[idx, 'Density']:,.0f}",
                       showarrow=True, arrowhead=0, arrowcolor='#AAAAAA', ax=0, ay=-25, font=dict(size=10, color='#555555'))
for cname in order:
    fig.add_annotation(x=cname, y=np.log10(spread.loc[cname, 'min']), text=f"median {med[cname]:.0f}",
                       showarrow=False, yshift=-14, font=dict(size=10, color='#555555'))

fig.update_layout(showlegend=False, title_x=0, title_font_size=18, margin=dict(t=90),
                  xaxis_title=None, yaxis_title='People per km², 2022 (log scale)')
fig.update_xaxes(showgrid=False)
fig.update_yaxes(gridcolor='#EEEEEE')
fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** **Choropleth map with binned (classed) colours.** "Where in the world" requires a map. Instead of a continuous scale I binned the % change into 6 ordered classes with a diverging blue-to-orange palette (CVD-safe, no red/green): bins make shrinking vs growing countries instantly distinguishable and stop Qatar's +317% from compressing everyone else's colours. A discrete legend replaces the colourbar.

In [14]:
# Q5
d = df.copy()
d['Change'] = (d['2022 Population'] / d['2000 Population'] - 1) * 100
bins = [-100, -10, 0, 10, 30, 60, 1000]
labels = ['Shrank > 10%', 'Shrank 0–10%', 'Grew 0–10%', 'Grew 10–30%', 'Grew 30–60%', 'Grew > 60%']
d['Class'] = pd.cut(d['Change'], bins=bins, labels=labels).astype(str)
colors = dict(zip(labels, ['#B35806', '#F1A340', '#C6DBEF', '#6BAED6', '#2171B5', '#08306B']))   # orange = shrank, blue = grew

n_shr = int((d['Change'] < 0).sum())
n_eu = int(((d['Change'] < 0) & (d['Continent'] == 'Europe')).sum())

fig = px.choropleth(d, locations='CCA3', locationmode='ISO-3', color='Class', hover_name='Country',
                    hover_data={'Change': ':.1f', 'Class': False, 'CCA3': False},
                    category_orders={'Class': labels}, color_discrete_map=colors, template='plotly_white',
                    title=f"Most of the world grew since 2000; the {n_shr} shrinking countries cluster in Eastern Europe<br>"
                          f"and the Baltics ({n_eu} of {n_shr} are European)")
fig.update_traces(marker_line_color='white', marker_line_width=0.3)
fig.update_geos(showframe=False, showcoastlines=False, projection_type='natural earth')
fig.update_layout(title_x=0, title_font_size=18, margin=dict(t=90, l=0, r=0, b=0),
                  legend=dict(title='Population change<br>2000 → 2022', orientation='v', x=0.01, y=0.3))
fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** **Treemap.** A two-level part-to-whole question (continent, then country) is what treemaps are for: area is population, nesting shows the hierarchy, and both levels fit in one figure. Colour encodes the continent (parents and their countries share a hue, lighter for countries), so no legend is needed. Tiny territories are too small to label and drop out visually, which is accurate: they are tiny.

In [15]:
# Q6
PAL = {'Africa': '#D55E00', 'Asia': '#0072B2', 'Europe': '#CC79A7',
       'North America': '#E69F00', 'South America': '#56B4E9', 'Oceania': '#666666'}

d = df.copy()
d['Pop (M)'] = d['2022 Population'] / 1e6
total = d['Pop (M)'].sum()
by_c = d.groupby('Continent')['Pop (M)'].sum().sort_values(ascending=False)

fig = px.treemap(d, path=[px.Constant('World'), 'Continent', 'Country'], values='Pop (M)', color='Continent',
                 color_discrete_map=PAL, template='plotly_white',
                 title=f"{by_c.index[0]} and {by_c.index[1]} together hold {by_c.iloc[:2].sum() / total:.0%} of the world's "
                       f"{total / 1e3:.1f} billion people;<br>China and India each have more people than Europe or the Americas")
fig.update_traces(root_color='white', marker_line_color='white', marker_line_width=2,
                  texttemplate='<b>%{label}</b><br>%{value:,.0f}M', textfont_size=13,
                  hovertemplate='%{label}: %{value:,.0f}M (%{percentRoot:.1%} of world)<extra></extra>')
fig.update_layout(title_x=0, title_font_size=18, margin=dict(t=90, l=10, r=10, b=10))
fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** **100% stacked bar chart (horizontal).** The question is about shares of a whole, so each year is one bar that always sums to 100%. Horizontal bars with years on the y-axis read like a timeline top to bottom (1970 at the top). Africa (blue) and Europe (orange) are the two stories and sit at opposite ends of the bar, so their growth and shrinkage is easy to follow; Asia and the rest are neutral greys, and labels are placed directly on the first bar instead of a legend.

In [16]:
# Q7
d = df_long.copy()
d['Group'] = np.where(d['Continent'].isin(['Africa', 'Asia', 'Europe']), d['Continent'], 'Rest of the world')
sh = d.groupby(['Year', 'Group'], as_index=False)['Population'].sum()
sh['Share'] = sh['Population'] / sh.groupby('Year')['Population'].transform('sum') * 100
sh['Year'] = sh['Year'].astype(str)
piv = sh.pivot(index='Year', columns='Group', values='Share')

order = ['Africa', 'Asia', 'Rest of the world', 'Europe']              # left -> right
colors = {'Africa': '#0B5394', 'Asia': '#9A9A9A', 'Rest of the world': '#D9D9D9', 'Europe': '#E69F00'}

fig = px.bar(sh, y='Year', x='Share', color='Group', orientation='h', text='Share', template='plotly_white',
             category_orders={'Group': order, 'Year': sorted(sh['Year'].unique())}, color_discrete_map=colors,
             title=f"Africa gained {piv.loc['2022', 'Africa'] - piv.loc['1970', 'Africa']:.0f} percentage points of the world's population "
                   f"since 1970;<br>Europe lost {piv.loc['1970', 'Europe'] - piv.loc['2022', 'Europe']:.0f}")
fig.update_traces(texttemplate='%{x:.0f}%', textposition='inside', insidetextanchor='middle', marker_line_width=0)
fig.update_traces(textfont_color='white', selector=dict(name='Africa'))

left = 0
for g in order:                                                         # direct labels above the first (1970) bar
    w = piv.loc['1970', g]
    fig.add_annotation(x=left + w / 2, y=-0.75, text=f"<b>{g}</b>", showarrow=False,
                       font=dict(size=12, color=colors[g] if g in ('Africa', 'Europe') else '#666666'))
    left += w

fig.update_layout(showlegend=False, title_x=0, title_font_size=18, margin=dict(t=100), xaxis_title=None, yaxis_title=None, bargap=0.2)
fig.update_xaxes(visible=False, range=[0, 100])
fig.update_yaxes(autorange='reversed', showgrid=False)
fig.show()